In [7]:
!pip install faker

In [13]:
# Re-import necessary libraries after execution reset
import pandas as pd
from faker import Faker
import random

# Initialize Faker
fake = Faker()

# Generating a fixed dataset based on the user's query requirements

# SALES table data generation
sales_data = {
    "Date": [
        "2023-03-18", "2023-03-18", "2023-01-10", "2023-01-15", "2023-02-20",
        "2022-06-05", "2022-11-10", "2022-12-20", "2022-05-15", "2023-03-18"
    ],
    "Order_id": [f"ORD{1000 + i}" for i in range(10)],
    "Item_id": [f"ITM{random.randint(1, 3)}" for _ in range(10)],
    "Customer_id": [f"CUST{random.randint(1, 5)}" for _ in range(10)],
    "Quantity": [random.randint(1, 5) for _ in range(10)],
    "Revenue": [round(random.uniform(50.0, 300.0), 2) for _ in range(10)],
}
sales_df = pd.DataFrame(sales_data)

# ITEMS table data generation
items_data = {
    "Item_id": [f"ITM{i+1}" for i in range(3)],
    "Item_name": ["WidgetA", "WidgetB", "WidgetC"],
    "Price": [150.0, 100.0, 50.0],
    "Department": ["Electronics", "Home", "Sports"]
}
items_df = pd.DataFrame(items_data)

# CUSTOMERS table data generation
customers_data = {
    "Customer_id": [f"CUST{i+1}" for i in range(5)],
    "First_name": ["John", "Alice", "Bob", "John", "Jane"],
    "Last_name": ["Doe", "Smith", "Brown", "Doe", "Davis"],
    "Address": [fake.address().replace('\n', ', ') for _ in range(5)]
}
customers_df = pd.DataFrame(customers_data)
# Display the data
print("SALES Table:")
display(sales_df)
print("\nITEMS Table:")
display(items_df)
print("\nCUSTOMERS Table:")
display(customers_df)


SALES Table:


,Date,Order_id,Item_id,Customer_id,Quantity,Revenue
0,2023-03-18,ORD1000,ITM3,CUST5,2,65.36
1,2023-03-18,ORD1001,ITM2,CUST3,2,209.50
2,2023-01-10,ORD1002,ITM1,CUST1,4,223.80
3,2023-01-15,ORD1003,ITM2,CUST1,1,52.24
4,2023-02-20,ORD1004,ITM2,CUST4,4,99.12
5,2022-06-05,ORD1005,ITM3,CUST4,5,296.94
6,2022-11-10,ORD1006,ITM2,CUST4,2,187.80
7,2022-12-20,ORD1007,ITM2,CUST4,3,78.93
8,2022-05-15,ORD1008,ITM2,CUST2,2,83.06
9,2023-03-18,ORD1009,ITM1,CUST4,5,155.42



ITEMS Table:


,Item_id,Item_name,Price,Department
0,ITM1,WidgetA,150.0,Electronics
1,ITM2,WidgetB,100.0,Home
2,ITM3,WidgetC,50.0,Sports



CUSTOMERS Table:


,Customer_id,First_name,Last_name,Address
0,CUST1,John,Doe,"746 Valerie Island Apt. 749, Andrewstown, FL 8..."
1,CUST2,Alice,Smith,"134 Mclean Shore Apt. 716, Jeffreystad, WV 08611"
2,CUST3,Bob,Brown,"Unit 5370 Box 8003, DPO AE 86911"
3,CUST4,John,Doe,"0948 Ronald Bypass Suite 637, Lake Robinland, ..."
4,CUST5,Jane,Davis,"392 Orr Squares Suite 711, East Melaniemouth, ..."


In [14]:
import sqlite3

In [15]:
conn = sqlite3.connect(":memory:")

In [16]:
# Load DataFrames as tables
sales_df.to_sql("SALES", conn, index=False, if_exists="replace")
items_df.to_sql("ITEMS", conn, index=False, if_exists="replace")
customers_df.to_sql("CUSTOMERS", conn, index=False, if_exists="replace")

5

In [18]:
# Query1: Pull total number of orders that were completed on 18th March 2023 with the first name ‘John’ and last name Doe’
query_1 = """
SELECT COUNT(*) AS Total_Orders
FROM SALES
INNER JOIN CUSTOMERS ON SALES.Customer_id = CUSTOMERS.Customer_id
WHERE SALES.Date = '2023-03-18'
AND CUSTOMERS.First_name = 'John'
AND CUSTOMERS.Last_name = 'Doe';
"""
result_1 = pd.read_sql_query(query_1, conn)
print(result_1)

   Total_Orders
0             1


In [22]:
# Query2: Pull total number of customers that purchased in January 2023 and the average amount spend per customer
query_2 = """
SELECT COUNT(DISTINCT Customer_id) AS Total_customers, AVG(Revenue) AS Average_amount_spend_per_customer
FROM SALES
WHERE SALES.Date BETWEEN '2023-01-01' AND '2023-01-31';
"""
result_2 = pd.read_sql_query(query_2, conn)
print(result_2)

   Total_customers  Average_amount_spend_per_customer
0                1                             138.02


In [25]:
#Query3: Pull the departments that generated less than $600 in 2022
query_3 = """
SELECT DISTINCT(Department)
FROM ITEMS
INNER JOIN SALES ON ITEMS.Item_id = SALES.Item_id
WHERE SALES.DATE LIKE '2022%' AND SALES.Revenue < 600;
"""
result_3 = pd.read_sql_query(query_3, conn)
print(result_3)

  Department
0     Sports
1       Home


In [28]:
#Query4: What is the most and least revenue we have generated by an order
query_4 = """
SELECT MAX(Revenue) AS MAX_REVENUE, MIN(Revenue) AS MIN_REVENUE
FROM SALES;
"""
result_4 = pd.read_sql_query(query_4, conn)
print(result_4)

   MAX_REVENUE  MIN_REVENUE
0       296.94        52.24


In [44]:
#Query5: What were the orders that were purchased in our most lucrative order
query_5 = """
SELECT S.Order_id, S.Date, S.Quantity, S.Revenue, I.Item_name
FROM SALES S
JOIN ITEMS I ON S.Item_id = I.Item_id
WHERE S.Revenue = (SELECT MAX(Revenue) FROM SALES);
"""
result_5 = pd.read_sql_query(query_5, conn)
print(result_5)


  Order_id        Date  Quantity  Revenue Item_name
0  ORD1005  2022-06-05         5   296.94   WidgetC
